# File System
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Hash Tables, Sorting, Trees · **Difficulty/Frequency:** Common (5/10)


## Concepts

**What this problem is really testing:**
- Hash-map accumulation — tallying total size per collection
- A size-bounded min-heap, for picking the top K without sorting everything
- Bottom-up propagation up a parent chain, for the hierarchy follow-up

**Why each one shows up here:**
- "Total size per collection" is a simple tally.
- "Top N collections" is the classic top-K problem — a size-N heap beats sorting everything when N is small.
- The hierarchy follow-up ("a collection's total should include its children's totals too") is the same bottom-up aggregation idea used elsewhere in this folder (Confluence Page Word Count, Company Hierarchy) — just walking **up** a parent pointer instead of recursing **down** a list of children.

**The one idea to hold onto:** accumulate exact totals in a single pass (a hash map), then pick the right tool for "top N" — sorting everything is correct, but wasteful once N is small compared to the total number of collections.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`, here `collections.defaultdict(int)`) stores key → value pairs by hashing the key to a slot, giving **O(1) average** insert/lookup/update.
- Used here to build up `collection_name -> total_size` in one pass over the files.

**What is a Heap (priority queue), and why keep it size-bounded?**
- A heap keeps the min (or max) element accessible in O(1), with O(log n) push/pop.
- A **size-N min-heap**, used for "give me the top N largest": push every candidate in, and whenever the heap grows past N, pop the smallest one out. By the end, the heap holds exactly the N largest — without ever fully sorting everything.
- **Cost:** O(C log N) for C candidates, vs. O(C log C) to sort them all — a real win once N is much smaller than C.
- **In Python:** the `heapq` module (a min-heap built on a plain list) — `heapq.heappush` / `heapq.heappop`.

**Bottom-up propagation via parent pointers.**
- When collections form a hierarchy (a collection can have a parent collection), a leaf collection's size needs to count toward every one of its ancestors' totals too.
- The fix: for each collection, walk `parent_map.get(current)` repeatedly all the way up, adding that collection's own size to every node visited along the way.
- This is the mirror image of "combine children into the parent" (post-order, walking down a tree) — here we're walking **up** instead.


## Problem Statement

Given files (each with a size, and zero or more collection tags), report: the total size of all files, and the **top N** collections by total size. Untagged files contribute to an overall "untagged" total, tracked separately (not as a collection named `"untagged"`, to avoid colliding with a real collection of that name).

**Follow-up 1:** a file can belong to multiple collections (`collectionIds: [...]`) -- add its size to every collection listed.
**Follow-up 2:** a collection can have a parent collection -- a child's total size must propagate up to every ancestor before ranking.

**Example:** `collection1` (2 files) -> 400; `collection2` (1 file) -> 300; untagged -> 110; top-2 -> `["collection1", "collection2"]`.


### Approach 1 -- Naive (sort all collections, take the first N)

**Idea:** tally every collection's total size in one pass (as any approach must), then `sorted()` the whole list of collections by size and slice the top N.

**Time complexity:** O(F) to tally (F = number of files, touching each file's collection list), then **O(C log C)** to sort all C collections -- even though only N of them are ultimately kept.

**Space complexity:** O(C) for the tally map.


In [ ]:
from typing import Dict, List, Optional
from collections import defaultdict


def top_collections_sorted(files: List[dict], n: int) -> List[str]:
    collection_sizes: Dict[str, int] = defaultdict(int)
    untagged_size = 0
    for f in files:
        colls = f.get("collectionIds", [])
        if not colls:
            untagged_size += f["size"]
        for c in colls:
            collection_sizes[c] += f["size"]

    ordered = sorted(collection_sizes.items(), key=lambda x: (-x[1], x[0]))   # O(C log C)
    return [name for name, _ in ordered[:n]]


### Approach 2 -- Optimal (size-N min-heap)

**Idea:** same tally pass, but instead of sorting everything, maintain a **min-heap bounded to size N**: push each `(size, name)` pair; if the heap grows past N, pop the smallest. After the pass, the heap holds exactly the N largest -- sort just those N (cheap) for the final descending order.

**Time complexity:** O(F) to tally, **O(C log N)** for the heap maintenance (C pushes/conditional-pops, each O(log N)), plus O(N log N) to sort the final N -- strictly better than O(C log C) whenever N << C.

**Space complexity:** O(C) for the tally map (unavoidable -- every collection's exact size must be computed), O(N) for the heap.


In [ ]:
import heapq


def top_collections(files: List[dict], n: int) -> List[str]:
    collection_sizes: Dict[str, int] = defaultdict(int)
    untagged_size = 0
    total_size = 0

    for f in files:
        total_size += f["size"]
        colls = f.get("collectionIds", [])
        if not colls:
            untagged_size += f["size"]
        for c in colls:
            collection_sizes[c] += f["size"]

    heap: List[tuple] = []
    for name, size in collection_sizes.items():
        heapq.heappush(heap, (size, name))
        if len(heap) > n:
            heapq.heappop(heap)          # discard the current smallest -- only the N largest survive

    top = sorted(heap, key=lambda x: (-x[0], x[1]))
    return [name for _, name in top]


def report(files: List[dict], n: int) -> dict:
    """Full report: total size, untagged size, and top-N collections."""
    collection_sizes: Dict[str, int] = defaultdict(int)
    untagged_size = 0
    total_size = 0
    for f in files:
        total_size += f["size"]
        colls = f.get("collectionIds", [])
        if not colls:
            untagged_size += f["size"]
        for c in colls:
            collection_sizes[c] += f["size"]
    return {
        "total_size": total_size,
        "untagged_size": untagged_size,
        "top_collections": top_collections(files, n),
        "collection_sizes": dict(collection_sizes),
    }


### Follow-up 2 -- Hierarchical collections (propagate sizes to ancestors)

**Idea:** given a `parent_map` (`collection -> parent_collection_or_None`), each collection's *own* size must also count toward every ancestor. For each collection, walk its parent chain, adding its own size to every collection visited (itself included). Rank using these **propagated** totals instead of the raw per-collection totals.

**Time complexity:** O(C * H) where H is the maximum hierarchy depth -- each of C collections walks up to H ancestors.

**Space complexity:** O(C) for the propagated-totals map.


In [ ]:
def propagate_to_ancestors(collection_sizes: Dict[str, int], parent_map: Dict[str, Optional[str]]) -> Dict[str, int]:
    result: Dict[str, int] = defaultdict(int)
    for name, size in collection_sizes.items():
        current: Optional[str] = name
        while current is not None:
            result[current] += size
            current = parent_map.get(current)
    return dict(result)


def top_collections_hierarchical(files: List[dict], parent_map: Dict[str, Optional[str]], n: int) -> List[str]:
    raw_sizes: Dict[str, int] = defaultdict(int)
    for f in files:
        for c in f.get("collectionIds", []):
            raw_sizes[c] += f["size"]

    propagated = propagate_to_ancestors(raw_sizes, parent_map)

    heap: List[tuple] = []
    for name, size in propagated.items():
        heapq.heappush(heap, (size, name))
        if len(heap) > n:
            heapq.heappop(heap)
    top = sorted(heap, key=lambda x: (-x[0], x[1]))
    return [name for _, name in top]


## Verification

Check against the worked example, both follow-ups, and the tie-breaking / untagged-collision edge cases the Talking Points call out.

In [ ]:
files = [
    {"file": "file1.txt", "size": 100},
    {"file": "file2.txt", "size": 200, "collectionIds": ["collection1"]},
    {"file": "file3.txt", "size": 200, "collectionIds": ["collection1"]},
    {"file": "file4.txt", "size": 300, "collectionIds": ["collection2"]},
    {"file": "file5.txt", "size": 10},
]

rep = report(files, n=2)
assert rep["total_size"] == 810
assert rep["untagged_size"] == 110
assert rep["collection_sizes"] == {"collection1": 400, "collection2": 300}
assert rep["top_collections"] == ["collection1", "collection2"]
assert top_collections_sorted(files, 2) == top_collections(files, 2)   # both approaches agree

# Follow-up 1: a file in MULTIPLE collections contributes its full size to each
files_multi = [
    {"file": "file1.txt", "size": 100},
    {"file": "file2.txt", "size": 200, "collectionIds": ["collection1"]},
    {"file": "file3.txt", "size": 200, "collectionIds": ["collection1"]},
    {"file": "file4.txt", "size": 300, "collectionIds": ["collection2", "collection3"]},
    {"file": "file5.txt", "size": 10},
]
rep_multi = report(files_multi, n=3)
assert rep_multi["collection_sizes"] == {"collection1": 400, "collection2": 300, "collection3": 300}
# collection2 and collection3 tie at 300 -- alphabetical tiebreak decides the order
assert rep_multi["top_collections"] == ["collection1", "collection2", "collection3"]

# Follow-up 2: collection2 is a CHILD of collection1 -- collection1's total must include collection2's
parent_map = {"collection1": None, "collection2": "collection1"}
files_hier = [
    {"file": "a.txt", "size": 100, "collectionIds": ["collection1"]},
    {"file": "b.txt", "size": 250, "collectionIds": ["collection2"]},
]
top_hier = top_collections_hierarchical(files_hier, parent_map, n=2)
assert top_hier == ["collection1", "collection2"]   # collection1: 100+250=350 > collection2: 250

raw = defaultdict(int)
for f in files_hier:
    for c in f["collectionIds"]:
        raw[c] += f["size"]
propagated = propagate_to_ancestors(dict(raw), parent_map)
assert propagated == {"collection1": 350, "collection2": 250}

# Edge cases
assert top_collections([], 5) == []
assert report([], 3)["total_size"] == 0
assert top_collections(files, 0) == []          # asking for zero collections
assert top_collections(files, 100) == ["collection1", "collection2"]   # n larger than available

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **A collection hierarchy that's a DAG (multiple parents), not a tree.** Naively walking every parent chain double-counts a shared ancestor's contribution from a descendant reachable via two paths. Fix with memoization (compute each collection's *own* contribution once, then propagate via a **topological order** of the DAG so every node is finalized only after all its children are) rather than the simple per-collection parent walk shown above.
- **Files added, removed, or re-tagged at runtime.** Maintain `collection_sizes` incrementally (add/subtract on each mutation) rather than recomputing from scratch; the heap-based top-N would then need to be rebuilt from the updated map on each query (O(C log N)) since a plain heap doesn't support efficient arbitrary-key decrease -- the same lazy-deletion trade-off discussed in the Content Popularity Tracker notebook.
- **Files too numerous to fit in memory.** A map-reduce-style approach: partition files across workers, have each worker produce its own partial `collection -> size` map, then merge partial maps (summing matching keys) before ranking -- merging is associative and commutative, so partial results can combine in any order.
- **Collection names as hierarchical path strings (`"a/b/c"`) instead of explicit parent pointers.** Parse each path into path.split("/") segments and either build an explicit parent map from consecutive segment pairs, or propagate directly using string prefixes (`"a/b/c"`'s size counts toward `"a/b"` and `"a"`) -- shown as a small variant below.
- **Top-N at multiple points in time (hourly snapshots).** Pre-aggregate into fixed time buckets and maintain a rolling top-N structure (e.g. re-run the size-N heap per bucket) rather than recomputing from all historical files on every snapshot.


In [ ]:
def propagate_path_collections(file_sizes_by_path: Dict[str, int]) -> Dict[str, int]:
    """Bonus: collections named as hierarchical paths, e.g. 'a/b/c' -- propagate to every ancestor prefix."""
    result: Dict[str, int] = defaultdict(int)
    for path, size in file_sizes_by_path.items():
        parts = path.split("/")
        for i in range(1, len(parts) + 1):
            ancestor = "/".join(parts[:i])
            result[ancestor] += size
    return dict(result)


demo = propagate_path_collections({"a/b/c": 100, "a/b": 50})
assert demo == {"a": 150, "a/b": 150, "a/b/c": 100}
print("Path-based hierarchy propagation:", demo)


## Empirical complexity check

`top_collections` (size-N heap) should beat `top_collections_sorted` (full sort) as the number of **distinct collections C** grows large relative to a small, fixed N -- O(C log N) vs. O(C log C).

| Growth when C doubles | Implies |
|---|---|
| ~2x (both) | roughly linear-ish for both, since log factors grow slowly -- watch the ABSOLUTE gap between them widen instead |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

N_TOP = 5   # fixed, small top-N -- the case the heap approach is designed for


def make_worst_case(num_collections):
    # One file per collection -- num_collections distinct collections, sizes all distinct.
    files = [{"file": f"f{i}.txt", "size": i + 1, "collectionIds": [f"coll{i}"]} for i in range(num_collections)]
    return (files, N_TOP)


solutions = {
    "sorted (O(C log C))": top_collections_sorted,
    "heap (O(C log N))": top_collections,
}
sizes = [4000, 8000, 16000, 32000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Size-bounded min-heap for top-K when K is small.** Push everything, pop the smallest whenever the heap exceeds K -- O(C log K) beats sorting (O(C log C)) whenever K << C, without ever materializing a full sorted order you'd throw most of away.
- **Track a special "doesn't fit the normal categories" bucket separately, not as a same-shaped entry.** Untagged files as a plain variable (not a collection literally named `"untagged"`) avoids a real collection with that name silently merging with it.
- **A "multiple tags per item" extension is just an inner loop over a list instead of a single value.** The multi-collection follow-up required zero structural change -- the single-tag version's `for c in colls` already generalizes.
- **Propagating up a parent chain is the same aggregation idea as combining down from children -- just walked in the other direction.** Compare to the post-order tree aggregation used in the Confluence Page Word Count and Company Hierarchy notebooks; here it's a flat parent-pointer walk instead of a recursive children list, but the "combine a leaf value into every ancestor" idea is identical.
- **Related problems:** Top K Frequent Elements (LeetCode 347, identical heap-vs-sort trade-off), Kth Largest Element in a Stream (the same bounded-heap idea maintained incrementally), any tag/label aggregation system (blog post tags, e-commerce category rollups).
- **Common pitfalls:** sorting everything when only a small top-K was asked for; conflating "untagged" as a category with an actual collection that happens to be named that; forgetting that a hierarchy with multiple parents (a DAG) breaks the simple single-parent-walk propagation and needs deduplication/topological handling instead.
